# Notebook 1: Training, Testing, and Saving the Whisper Model

This notebook covers:
- Extracting and loading the LibriSpeech dataset
- Preprocessing audio files
- Fine-tuning OpenAI Whisper (base) using HuggingFace Transformers
- Training on `train-clean-100`, validating on `dev-clean`, testing on `test-clean`
- Saving trained model weights to `model/module_1/weights/`

## Step 1: Install Dependencies

In [15]:
%pip install transformers datasets torch torchaudio accelerate evaluate jiwer soundfile librosa --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports and Configuration

In [16]:
import os
import tarfile
import torch
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import Dataset, Audio
import evaluate

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path("../../../")         # NLP_project root (where .tar.gz files are)
DATA_DIR    = BASE_DIR
EXTRACT_DIR = BASE_DIR / "librispeech" # extracted LibriSpeech goes here
WEIGHTS_DIR = Path("../weights")        # model is saved here
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME        = "openai/whisper-base"
LANGUAGE          = "english"
TASK              = "transcribe"
SAMPLE_RATE       = 16_000
MAX_TRAIN_SAMPLES = 500   # set None to use the full dataset (much slower)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device        : {device}")
print(f"Weights dir   : {WEIGHTS_DIR.resolve()}")

Device        : cpu
Weights dir   : E:\UMD\Data_641_PCS1\NLP_project\PronounceAI\model\module_1\weights


In [17]:
print(EXTRACT_DIR.resolve())

E:\UMD\Data_641_PCS1\NLP_project\PronounceAI\librispeech


## Step 3: Extract LibriSpeech Archives

In [ ]:
def extract_archive(tar_path: Path, dest: Path) -> None:
    """Extract a .tar.gz archive once; skip if already extracted."""
    if dest.exists():
        print(f"Already extracted: {tar_path.name}")
    else:
        print(f"Extracting {tar_path.name} ...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(dest)
        print("Done.")
        
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
extract_archive(DATA_DIR / "train-clean-100.tar.gz", EXTRACT_DIR)
extract_archive(DATA_DIR / "dev-clean.tar.gz",       EXTRACT_DIR)
extract_archive(DATA_DIR / "test-clean.tar.gz",      EXTRACT_DIR)

Extracting train-clean-100.tar.gz ...
Done.
Extracting dev-clean.tar.gz ...
Done.
Extracting test-clean.tar.gz ...
Done.


## Step 4: Load LibriSpeech into HuggingFace Dataset

In [28]:
import soundfile as sf
import librosa

def parse_librispeech_split(root: Path, split_name: str) -> Dataset:
    """
    Walk a LibriSpeech split and collect (audio_path, transcript) pairs.
    Layout: root/LibriSpeech/<split>/<speaker>/<chapter>/*.flac + *.trans.txt
    """
    audio_paths, transcripts = [], []
    split_dir = root / "LibriSpeech" / split_name
    if not split_dir.exists():
        raise FileNotFoundError(f"Split directory not found: {split_dir}")

    for trans_file in sorted(split_dir.rglob("*.trans.txt")):
        chapter_dir = trans_file.parent
        with open(trans_file, "r") as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                if len(parts) != 2:
                    continue
                utt_id, text = parts
                flac_path = chapter_dir / f"{utt_id}.flac"
                if flac_path.exists():
                    audio_paths.append(str(flac_path))
                    transcripts.append(text.lower())

    # Store raw file paths — audio is decoded manually in preprocess()
    ds = Dataset.from_dict({"audio_path": audio_paths, "sentence": transcripts})
    return ds

print("Loading splits...")
raw_train = parse_librispeech_split(EXTRACT_DIR, "train-clean-100")
raw_dev   = parse_librispeech_split(EXTRACT_DIR, "dev-clean")
raw_test  = parse_librispeech_split(EXTRACT_DIR, "test-clean")

if MAX_TRAIN_SAMPLES:
    raw_train = raw_train.select(range(min(MAX_TRAIN_SAMPLES, len(raw_train))))
    raw_dev   = raw_dev.select(range(min(100, len(raw_dev))))

print(f"Train : {len(raw_train)} samples")
print(f"Dev   : {len(raw_dev)} samples")
print(f"Test  : {len(raw_test)} samples")

Loading splits...
Train : 500 samples
Dev   : 100 samples
Test  : 2620 samples


## Step 5: Load Whisper Processor and Preprocess Data

Each sample is processed **one at a time** (`batched=False`).
- Audio → log-mel spectrogram (80 × 3000)
- Text  → token IDs (padding handled later by the DataCollator)

In [29]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

def preprocess(sample):
    """
    Load audio from disk and convert to model inputs.
    Uses soundfile directly to avoid the torchcodec dependency.
    """
    audio_array, sr = sf.read(sample["audio_path"], dtype="float32")

    # Resample if needed (LibriSpeech is 16kHz, so usually a no-op)
    if sr != SAMPLE_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)

    # Log-mel spectrogram — shape (80, 3000), stored as numpy
    features = processor.feature_extractor(
        audio_array,
        sampling_rate=SAMPLE_RATE,
        return_tensors="np",
    )
    sample["input_features"] = features.input_features[0]  # (80, 3000)

    # Token IDs — plain Python list, no padding yet
    sample["labels"] = processor.tokenizer(
        sample["sentence"],
        truncation=True,
        max_length=448,
    ).input_ids

    return sample

print("Preprocessing train split...")
train_dataset = raw_train.map(
    preprocess,
    remove_columns=raw_train.column_names,
    desc="Train",
)
print("Preprocessing dev split...")
dev_dataset = raw_dev.map(
    preprocess,
    remove_columns=raw_dev.column_names,
    desc="Dev",
)
print("Done.")

Preprocessing train split...


Train: 100%|██████████| 500/500 [00:06<00:00, 77.53 examples/s] 


Preprocessing dev split...


Dev: 100%|██████████| 100/100 [00:01<00:00, 94.11 examples/s]

Done.


## Step 6: Data Collator

The collator pads each batch and replaces padding token IDs with `-100`
so they are ignored by the cross-entropy loss.

In [30]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # ── Pad input features (log-mel spectrograms) ──────────────────────
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # ── Pad token sequences and replace padding with -100 ──────────────
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        # Mask padding positions so loss ignores them
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove decoder start token if it was prepended
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("DataCollator ready.")

DataCollator ready.


## Step 7: Evaluation Metric (WER)

In [22]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 back to pad token so decode() doesn't crash
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

## Step 8: Load Pretrained Whisper Model

In [23]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Disable forced decoding settings that interfere with fine-tuning
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False   # required when gradient_checkpointing=True

model = model.to(device)
print("Model loaded.")

Loading weights: 100%|██████████| 245/245 [00:00<00:00, 7332.07it/s]

Model loaded.


## Step 9: Training Arguments

In [24]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(WEIGHTS_DIR),

    # ── Batch & memory ─────────────────────────────────────────────────────
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,          # effective batch size = 8
    gradient_checkpointing=True,            # saves GPU memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=torch.cuda.is_available(),         # half-precision on GPU only

    # ── Learning rate ──────────────────────────────────────────────────────
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=200,                          # increase for better results

    # ── Evaluation & saving ────────────────────────────────────────────────
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,                # lower WER is better

    # ── Generation ─────────────────────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=225,

    # ── Misc ───────────────────────────────────────────────────────────────
    report_to="none",
    push_to_hub=False,
)

## Step 10: Train the Model

In [25]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,   # used for saving alongside model
)

print("Starting fine-tuning...")
train_result = trainer.train()
print("\nTraining complete.")
print(train_result.metrics)

Starting fine-tuning...


c:\Users\ateeq\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Wer
50,0.730421,0.350550,0.113200
100,0.146134,0.274155,0.107100
150,0.069325,0.275524,0.106500
200,0.051032,0.275951,0.101000


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para


Training complete.
{'train_runtime': 3633.1572, 'train_samples_per_second': 0.44, 'train_steps_per_second': 0.055, 'total_flos': 1.0299767390208e+17, 'train_loss': 0.41133365631103513, 'epoch': 3.176}


## Step 11: Evaluate on test-clean

In [26]:
print("Preprocessing test split...")
test_dataset = raw_test.map(
    preprocess,
    remove_columns=raw_test.column_names,
    desc="Test",
)

print("Evaluating on test-clean...")
test_results = trainer.evaluate(test_dataset)
print("\nTest results:")
for k, v in test_results.items():
    print(f"  {k}: {v}")

Preprocessing test split...


Test: 100%|██████████| 2620/2620 [00:49<00:00, 52.47 examples/s]
c:\Users\ateeq\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Evaluating on test-clean...


Training Loss,Validation Loss,Step,Wer
0.051032,0.192321,200,0.071600



Test results:
  eval_loss: 0.1923212856054306
  eval_wer: 0.0716


## Step 12: Save Model Weights and Processor

In [27]:
save_path = WEIGHTS_DIR / "whisper-base-librispeech"
trainer.save_model(str(save_path))
processor.save_pretrained(str(save_path))
print(f"Model and processor saved to:\n  {save_path.resolve()}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]

Model and processor saved to:
  E:\UMD\Data_641_PCS1\NLP_project\PronounceAI\model\module_1\weights\whisper-base-librispeech


---
## Done!

The fine-tuned model is saved in `model/module_1/weights/whisper-base-librispeech/`.

**Next:** Open **Notebook 2** to run evaluation and compute WER/CER on the test set.